# Retail Sales & Customer Behavior Analysis — Realmart
**Goal:** find which branches, cities and product lines earn the most, how customers pay, and what affects customer rating.

> **How to use:** put your instructor's dataset in the same folder as this notebook, change the file name in Step 1, then click *Run All*.

## Step 1: Import the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Change this file name to your instructor's dataset
df = pd.read_csv("Realmart_Sample_Data.csv")
df.columns = df.columns.str.strip()      # remove accidental spaces in column names
print("Rows, Columns:", df.shape)

## Step 2: First and last five records

In [ ]:
display(df.head())
display(df.tail())

## Step 3: Dataset information and summary statistics

In [ ]:
df.info()
display(df.describe())

## Step 4: Missing values
Numbers are filled with the **median**, text columns with the **most common value (mode)**.

In [ ]:
print("Missing values before:")
print(df.isnull().sum()[df.isnull().sum() > 0])

for col in df.columns:
    if df[col].isnull().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

print("\nTotal missing values after:", df.isnull().sum().sum())

## Step 5: Duplicate records

In [ ]:
print("Duplicates found:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Rows after removing duplicates:", len(df))

## Step 6: Standardize the Date column
The file has two date styles: `03/25/2019` (month/day/year) and `25-03-2019` (day-month-year). We read each style separately and save one format: `YYYY-MM-DD`.

In [ ]:
def fix_date(x):
    x = str(x).strip()
    try:
        if "/" in x:
            return pd.to_datetime(x, format="%m/%d/%Y")
        elif "-" in x:
            return pd.to_datetime(x, format="%d-%m-%Y")
    except ValueError:
        pass
    return pd.to_datetime(x, errors="coerce")   # anything else

df["Date"] = df["Date"].apply(fix_date)
print("Dates that could not be read:", df["Date"].isnull().sum())
df["Date"].head()

## Step 7: Fix data types

In [ ]:
for col in ["Branch", "City", "Customer type", "Gender", "Product line", "Payment"]:
    df[col] = df[col].astype("category")

df["Month"] = df["Date"].dt.month_name()
df.dtypes

## Step 8: Exploratory Data Analysis (EDA)

In [ ]:
print("Total revenue:", round(df["Total"].sum(), 2))
print("Average bill:", round(df["Total"].mean(), 2))
print("Average rating:", round(df["Rating"].mean(), 2))
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print()
for col in ["Branch", "City", "Customer type", "Gender", "Payment"]:
    print(df[col].value_counts(), "\n")

## Step 9: GroupBy analysis

In [ ]:
sales_branch  = df.groupby("Branch", observed=True)["Total"].sum().sort_values(ascending=False)
sales_city    = df.groupby("City", observed=True)["Total"].sum().sort_values(ascending=False)
sales_product = df.groupby("Product line", observed=True)["Total"].sum().sort_values(ascending=False)
rating_branch = df.groupby("Branch", observed=True)["Rating"].mean().sort_values(ascending=False)
payment_count = df["Payment"].value_counts()

print("Total Sales by Branch:\n", sales_branch.round(2), "\n")
print("Total Sales by City:\n", sales_city.round(2), "\n")
print("Total Sales by Product Line:\n", sales_product.round(2), "\n")
print("Average Rating by Branch:\n", rating_branch.round(2), "\n")
print("Payment Method Usage:\n", payment_count)

## Step 10: Correlation analysis

In [ ]:
# Tax, cogs and gross income are just calculated from Total, so they would always show ~1.0.
# We keep only the independent columns to get a meaningful result.
num = df[["Unit price", "Quantity", "Total", "Rating"]]
corr = num.corr()

# strongest pairs (ignore a column with itself)
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().sort_values(key=abs, ascending=False)
print("Top 5 strongest correlations:")
print(pairs.head(5).round(3))

print("\nCorrelation of each column with Rating:")
print(corr["Rating"].drop("Rating").round(3))

## Step 11: Visualizations (7 charts)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sales_branch.plot(kind="bar", color="steelblue", ax=ax[0], title="Total Sales by Branch")
sales_city.plot(kind="bar", color="seagreen", ax=ax[1], title="Total Sales by City")
for a in ax: a.set_ylabel("Sales"); a.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.savefig("chart1_branch_city.png", dpi=120); plt.show()

In [ ]:
plt.figure(figsize=(9, 4))
sales_product.sort_values().plot(kind="barh", color="coral")
plt.title("Total Sales by Product Line"); plt.xlabel("Sales")
plt.tight_layout(); plt.savefig("chart2_product_line.png", dpi=120); plt.show()

In [ ]:
plt.figure(figsize=(5, 5))
payment_count.plot(kind="pie", autopct="%1.1f%%", startangle=90)
plt.title("Payment Methods"); plt.ylabel("")
plt.tight_layout(); plt.savefig("chart3_payment_pie.png", dpi=120); plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(df["Rating"], bins=15, kde=True, color="purple")
plt.title("Distribution of Customer Rating")
plt.tight_layout(); plt.savefig("chart4_rating_hist.png", dpi=120); plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(x="Branch", y="Rating", data=df)
plt.title("Rating by Branch")
plt.tight_layout(); plt.savefig("chart5_rating_box.png", dpi=120); plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.tight_layout(); plt.savefig("chart6_heatmap.png", dpi=120); plt.show()

In [ ]:
daily = df.groupby("Date")["Total"].sum()
plt.figure(figsize=(10, 4))
daily.plot(color="darkorange")
plt.title("Daily Sales Trend"); plt.ylabel("Sales")
plt.tight_layout(); plt.savefig("chart7_daily_sales.png", dpi=120); plt.show()

## Step 12: Save the cleaned dataset

In [ ]:
df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")     # one consistent date format
df.to_csv("Retailmart_Cleaned_Dataset.csv", index=False)
df["Date"] = pd.to_datetime(df["Date"])
print("Saved: Retailmart_Cleaned_Dataset.csv")

## Step 13: Answers, insights and recommendations
This cell reads the results above and writes the answers automatically, so the text always matches your data.

In [ ]:
top_branch, top_city, top_prod = sales_branch.index[0], sales_city.index[0], sales_product.index[0]
low_prod = sales_product.index[-1]
top_pay = payment_count.index[0]
best_rating_branch, worst_rating_branch = rating_branch.index[0], rating_branch.index[-1]
best_pair = pairs.index[0]
rating_corr = corr["Rating"].drop("Rating").abs().max()
member_avg = df.groupby("Customer type", observed=True)["Total"].mean()
peak_day = daily.idxmax().date()
total = df["Total"].sum()

print("ANSWERS TO ANALYSIS QUESTIONS")
print(f"1. Highest revenue branch: {top_branch} ({sales_branch.iloc[0]:,.0f})")
print(f"2. City with most sales: {top_city} ({sales_city.iloc[0]:,.0f})")
print(f"3. Best product line: {top_prod} ({sales_product.iloc[0]:,.0f})")
print(f"4. Most used payment method: {top_pay} ({payment_count.iloc[0]} transactions)")
print(f"5. Highest average rating: Branch {best_rating_branch} ({rating_branch.iloc[0]:.2f})")
print(f"6. Strongest correlation: {best_pair[0]} & {best_pair[1]} ({pairs.iloc[0]:.2f})")

print("\nEIGHT BUSINESS INSIGHTS")
insights = [
 f"Branch {top_branch} earns the most revenue ({sales_branch.iloc[0]/total:.1%} of total sales).",
 f"{top_city} is the top city, while {sales_city.index[-1]} is the lowest.",
 f"'{top_prod}' is the best product line; '{low_prod}' is the weakest.",
 f"The gap between best and worst product line is {sales_product.iloc[0]-sales_product.iloc[-1]:,.0f}, so inventory should follow demand.",
 f"{top_pay} is the most popular payment method, so it must always work smoothly.",
 f"Branch {best_rating_branch} has the happiest customers ({rating_branch.iloc[0]:.2f}); Branch {worst_rating_branch} is lowest ({rating_branch.iloc[-1]:.2f}).",
 f"{best_pair[0]} and {best_pair[1]} are the most strongly linked ({pairs.iloc[0]:.2f}); Total is simply Unit price x Quantity (+ tax).",
 f"Rating has weak correlation with other numbers (max {rating_corr:.2f}), so price and quantity do not explain satisfaction. Service quality likely does.",
]
for i, t in enumerate(insights, 1): print(f"{i}. {t}")

print("\nBUSINESS CHALLENGES")
print(f"- Uneven sales across branches/cities. - Weak product line: {low_prod}. - Low rating at Branch {worst_rating_branch}. - Satisfaction is not explained by the data we collect.")

print("\nFIVE RECOMMENDATIONS")
recs = [
 f"Stock more '{top_prod}' and review or discount '{low_prod}'.",
 f"Copy what Branch {top_branch} does well (staffing, layout) in the other branches.",
 f"Train staff and run customer surveys at Branch {worst_rating_branch} to raise ratings.",
 f"Offer small rewards for {top_pay} payments and keep all payment methods reliable.",
 f"Add more staff at peak times (peak sales day was {peak_day}) and run promotions to lift weak cities.",
]
for i, t in enumerate(recs, 1): print(f"{i}. {t}")

print("\nTOP 3 RECOMMENDATIONS FOR MANAGEMENT: #1 (stock), #3 (service), #2 (best practice sharing)")